# 05 — Iterators and Generators

`for` is a thin wrapper over two method calls. Once you can implement the
iterator protocol by hand, generators stop being magic — they are just a
compiler-written iterator whose state is a suspended stack frame. This
lesson goes all the way to `send`, `throw`, `close`, and `yield from`,
which is also the machinery `await` is built on.

`#iterators` `#generators` `#coroutines` `#laziness`

<sub>Setup cell — run it once per session so `check()` works.</sub>

In [ ]:
# Wire the grader up. Run this cell first, every session.
import pathlib
import sys

_here = pathlib.Path.cwd()
_root = next((p for p in (_here, *_here.parents)
              if (p / "tutorial" / "grader.py").exists()), None)
if _root is None:
    raise RuntimeError("run this notebook from inside the tutorial repository")
sys.path.insert(0, str(_root))

from tutorial.grader import check, hint, solution   # noqa: E402

# A notebook kernel already owns an event loop, so asyncio.run() would refuse
# to start another one. Hand those coroutines a private loop on a worker
# thread instead, so the asyncio lessons run here exactly as they do in a
# script. (Outside a notebook this wrapper is a no-op.)
import asyncio as _asyncio                                    # noqa: E402
from concurrent.futures import ThreadPoolExecutor as _Pool    # noqa: E402

_plain_run = _asyncio.run


def _run_anywhere(coro, **kwargs):
    try:
        _asyncio.get_running_loop()
    except RuntimeError:
        return _plain_run(coro, **kwargs)
    with _Pool(1) as pool:
        return pool.submit(_plain_run, coro, **kwargs).result()


_asyncio.run = _run_anywhere

print("grader ready — Python", sys.version.split()[0])

## The protocol

- **iterable** — has `__iter__()` returning an iterator
- **iterator** — has `__next__()` and an `__iter__()` returning `self`

`for x in obj:` desugars to roughly:

```python
it = iter(obj)              # calls obj.__iter__()
while True:
    try:
        x = next(it)        # calls it.__next__()
    except StopIteration:
        break
    ...body...
```

Every iterator is an iterable; the reverse is false. A `list` is an
iterable that hands out a *fresh* iterator each time — which is why
you can loop over it twice.

In [ ]:
nums = [1, 2, 3]
it1, it2 = iter(nums), iter(nums)
print("independent iterators:", next(it1), next(it1), "|", next(it2))
print("iterator is its own iterable:", iter(it1) is it1)

# An iterator, hand-rolled.
class Countdown:
    def __init__(self, start): self.start = start
    def __iter__(self):
        return CountdownIterator(self.start)     # fresh state each time

class CountdownIterator:
    def __init__(self, n): self.n = n
    def __iter__(self): return self
    def __next__(self):
        if self.n <= 0:
            raise StopIteration
        self.n -= 1
        return self.n + 1

c = Countdown(3)
print("pass 1:", list(c))
print("pass 2:", list(c), "  <- reusable because __iter__ makes a new one")

Splitting *iterable* from *iterator* is what makes a container
re-iterable. If a class returns `self` from `__iter__` and mutates
its own state in `__next__`, it is single-use — like a file object
or a generator.

### ❓ Predict the output

Read the cell below and decide what it prints **before** running it. Then run it.

In [ ]:
class OneShot:
    def __init__(self, n): self.n = n
    def __iter__(self): return self
    def __next__(self):
        if self.n <= 0: raise StopIteration
        self.n -= 1
        return self.n

s = OneShot(3)
print(list(s))
print(list(s))
print(any(x > 100 for x in OneShot(3)))

<details>
<summary><b>the answer, and why</b></summary>

```
[2, 1, 0]
[]
False
```

`__iter__` returning `self` means the second `list()` resumes an
already-exhausted iterator and gets nothing — no error, just empty.
Silent emptiness is the signature bug of one-shot iterables.

</details>

## `iter()` has a second form

`iter(callable, sentinel)` builds an iterator that calls `callable()`
repeatedly until it returns `sentinel`. It is the neat way to drain
anything with a "read one" API.

In [ ]:
import io

stream = io.StringIO("alpha\nbeta\ngamma\n")
# Call readline() until it returns "" (EOF).
for line in iter(stream.readline, ""):
    print("line:", line.strip())

# Same trick over a queue-like object.
pending = [3, 1, 4, None, 5]
pop = iter(pending).__next__
print("until None:", list(iter(pop, None)))

## Generators: iterators the compiler writes for you

A function containing `yield` is a **generator function**. Calling it
runs no body code at all — it returns a generator object. Each `next()`
runs to the following `yield` and suspends, keeping locals and the
instruction pointer alive.

In [ ]:
import inspect

def gen():
    print("  [body starts]")
    x = yield 1
    print("  [resumed with]", x)
    yield 2
    print("  [body ends]")

g = gen()
print("calling gen() ran nothing:", type(g).__name__, inspect.getgeneratorstate(g))
print("next ->", next(g), inspect.getgeneratorstate(g))
print("next ->", next(g))
try:
    next(g)
except StopIteration as stop:
    print("StopIteration, value =", stop.value,
          "| state:", inspect.getgeneratorstate(g))

States are `GEN_CREATED` → `GEN_SUSPENDED` → `GEN_RUNNING` →
`GEN_CLOSED`. `inspect.getgeneratorstate` and
`inspect.getgeneratorlocals` are excellent debugging tools that
almost nobody knows about.

In [ ]:
# `return` in a generator sets StopIteration.value -- how a
# generator reports a final result rather than a stream item.
def parse(tokens):
    collected = []
    for tok in tokens:
        if tok == "STOP":
            return len(collected)          # becomes .value
        collected.append(tok)
        yield tok.upper()
    return len(collected)

g = parse(["a", "b", "STOP", "c"])
print("yielded:", list(g))

g2 = parse(["a", "b", "STOP", "c"])
try:
    while True:
        next(g2)
except StopIteration as stop:
    print("returned value:", stop.value)

`list(g)` throws the return value away — only `yield from` and
manual `next()` can see it. `await` uses exactly this channel to
deliver a coroutine's result.

### 🧠 Quiz

What does calling a generator function do?

- **a)** Runs the body until the first `yield`
- **b)** Runs the entire body and collects the yields into a list
- **c)** Runs nothing; it returns a generator object
- **d)** Raises unless you iterate it immediately

<details>
<summary><b>answer</b></summary>

**c** — Not a single statement of the body executes until the first `next()`. This is why validation in a generator function silently does nothing until someone iterates.

</details>

## `yield from`: delegation, not just a loop

`yield from sub` is *almost* `for x in sub: yield x`, but it also:

- forwards `send()` values into the sub-generator
- forwards `throw()` exceptions into it
- propagates `close()`
- evaluates to the sub-generator's **return value**

That full-duplex delegation is what made `asyncio` possible before
`async`/`await` had syntax.

In [ ]:
def leaf(name):
    total = 0
    while True:
        got = yield
        if got is None:
            return total          # finished: report the sum
        print(f"  {name} received {got}")
        total += got

def middle():
    subtotal = yield from leaf("leaf")     # captures the return value
    print("  middle saw subtotal:", subtotal)
    yield subtotal

m = middle()
next(m)                 # prime it
m.send(10)
m.send(5)
print("final:", m.send(None))

`send(x)` resumes the generator, makes the paused `yield` evaluate
to `x`, and returns the next yielded value. `next(g)` is exactly
`g.send(None)` — which is why a generator must be *primed* before
you can send it a real value.

In [ ]:
# Flattening arbitrarily nested structures -- the classic yield from.
def flatten(obj):
    if isinstance(obj, (str, bytes)):
        yield obj                       # strings are iterable: stop here
        return
    try:
        iterator = iter(obj)
    except TypeError:
        yield obj
        return
    for item in iterator:
        yield from flatten(item)

nested = [1, [2, [3, [4, [5]]]], "text", (6, {7, 8})]
print(list(flatten(nested)))

## `throw` and `close`: exceptions travel inward

- `g.throw(Exc)` raises `Exc` **at the paused `yield`**, so the
  generator can catch it and carry on.
- `g.close()` throws `GeneratorExit` there. The generator must not
  yield again — doing so raises `RuntimeError`.
- `finally` blocks in a generator run on close, which is what makes
  `@contextmanager` (lesson 11) work.

In [ ]:
def resilient():
    results = []
    try:
        while True:
            try:
                item = yield
                results.append(item)
            except ValueError as exc:
                print("  caught and continuing:", exc)
    finally:
        print("  cleanup ran, collected:", results)

g = resilient()
next(g)
g.send("a")
g.throw(ValueError("bad item"))     # generator survives
g.send("b")
g.close()                           # triggers the finally
print("state after close:", g.gi_frame is None)

In [ ]:
# Garbage collection also closes generators -- so `finally` runs
# even if you just walk away.
def leaky():
    try:
        yield 1
        yield 2
    finally:
        print("  finally ran during cleanup")

g = leaky()
print("got:", next(g))
del g                    # refcount hits zero -> close() -> finally
print("done")

Do not rely on this for releasing scarce resources: on PyPy or with
reference cycles it may be delayed. Use a `with` block *inside* the
generator, or `contextlib.closing`, when the cleanup matters.

### 🧠 Quiz

A paused generator receives `g.close()`. What happens inside?

- **a)** `StopIteration` is raised at the paused `yield`
- **b)** `GeneratorExit` is raised at the paused `yield`; `finally` blocks run
- **c)** The frame is discarded immediately with no cleanup
- **d)** `__del__` is called on the generator

<details>
<summary><b>answer</b></summary>

**b** — `GeneratorExit` inherits from `BaseException` — so a bare `except Exception:` inside the generator will not swallow it.

</details>

## Generator pipelines

Chained generators form a streaming pipeline: each stage pulls one
item at a time from the previous one. Memory stays flat regardless of
input size, and any stage can stop the whole chain early.

In [ ]:
def read_lines(text):
    for line in text.splitlines():
        yield line

def strip_comments(lines):
    for line in lines:
        stripped = line.split("#", 1)[0].strip()
        if stripped:
            yield stripped

def parse_pairs(lines):
    for line in lines:
        key, _, value = line.partition("=")
        yield key.strip(), value.strip()

def coerce(pairs):
    for key, value in pairs:
        if value.isdigit():
            value = int(value)
        elif value.lower() in {"true", "false"}:
            value = value.lower() == "true"
        yield key, value

config_text = '''
# server settings
host = localhost
port = 8080          # the http port
debug = true

retries = 3
'''

pipeline = coerce(parse_pairs(strip_comments(read_lines(config_text))))
print(dict(pipeline))

Each stage is independently testable and reusable, nothing is
buffered, and swapping `read_lines(text)` for a file object streams
a 100 GB config without changing another line.

## `itertools.tee`, and why you usually should not

Need two passes over a one-shot iterator? `tee` buffers everything one
branch has seen but the other has not. If the branches drift far apart,
`tee` quietly holds the entire stream in memory. Often `list()` is more
honest.

In [ ]:
from itertools import tee

source = iter(range(6))
a, b = tee(source, 2)
print("branch a:", list(a))
print("branch b:", list(b), "  <- replayed from tee's buffer")
print("source is exhausted:", list(source))

# Do not touch the original after tee-ing:
src2 = iter(range(6))
c, d = tee(src2, 2)
next(src2)                    # steals an item from under tee
print("after stealing:", list(c), list(d))

---

## ⚒ Exercise — A peekable iterator wrapper  
`05-iterators/peekable` · difficulty: **medium**

Implement `Peekable(iterable)` — an iterator that can look one item
ahead without consuming it. This is the core primitive of every
hand-written parser.

- `peek()` → the next item without advancing
- `peek(default)` → `default` instead of raising at the end
- `peek()` at the end with no default raises `StopIteration`
- `next(p)` returns the peeked item, then moves on
- `bool(p)` → True while items remain (must not consume one)
- it is an iterator: `iter(p) is p`, works in a `for` loop
- it must never pull more than one item ahead of the consumer

Edit the next cell, run it, then run the check cell below it.

In [ ]:
_MISSING = object()


class Peekable:
    def __init__(self, iterable):
        ...

In [ ]:
check("05-iterators/peekable")

<details>
<summary><b>hints</b></summary>

1. Keep a `_cache` slot holding either a sentinel or the buffered item.
2. `None` is a legitimate stream value — use a unique sentinel object, not `None`, to mean 'nothing buffered'.
3. `__bool__` is easiest as `try: self.peek() / except StopIteration: return False`. Passing your own sentinel as the *default* will not work — think about what `default is _MISSING` then evaluates to.
4. In `__next__`, return the cached item and reset the cache *before* returning, or you will hand out the same item twice.

</details>

<details>
<summary><b>reference solution</b></summary>

```python
_MISSING = object()

class Peekable:
    def __init__(self, iterable):
        self._it = iter(iterable)
        self._cache = _MISSING

    def __iter__(self):
        return self

    def __next__(self):
        if self._cache is not _MISSING:
            item, self._cache = self._cache, _MISSING
            return item
        return next(self._it)

    def peek(self, default=_MISSING):
        if self._cache is _MISSING:
            try:
                self._cache = next(self._it)
            except StopIteration:
                if default is _MISSING:
                    raise
                return default
        return self._cache

    def __bool__(self):
        try:
            self.peek()
        except StopIteration:
            return False
        return True
```

</details>

---

## ⚒ Exercise — A send()-driven running aggregator  
`05-iterators/running-agg` · difficulty: **brutal**

Write the generator `aggregator()` — a *coroutine* that consumes
numbers via `send()` and yields the running state after each one.

State dict: `{"count", "total", "mean", "min", "max", "error"}`.

Protocol:
- after priming with `next(agg)`, each `agg.send(x)` returns the
  updated state; `mean`/`min`/`max` are `None` while `count == 0`
- sending `"reset"` clears everything back to the empty state
- sending a non-numeric value leaves the statistics **unchanged**
  and yields a state whose `error` is `"cannot aggregate <repr>"`;
  the next good value clears `error` back to `None`
- `bool` must be rejected as non-numeric even though
  `isinstance(True, int)` is True
- `agg.throw(ValueError)` is caught inside: the generator yields
  its current state and keeps going
- on `close()`, a `finally` block appends the final count to the
  module-level list `CLOSED_WITH`

Then write `feed(agg, values)` that primes if needed, sends every
value, and returns the last state dict.

> **Why report the error through the yielded value instead of
> raising it?** Because an exception that propagates *out* of a
> generator frame terminates that generator for good — there is no
> "raise into the caller and stay alive". `throw()` works in the
> other direction: it raises *at* the paused `yield`, where an
> `except` inside the body can absorb it. Keeping a coroutine
> alive across bad input therefore means yielding the error, not
> raising it.

Edit the next cell, run it, then run the check cell below it.

In [ ]:
CLOSED_WITH = []


def aggregator():
    ...


def feed(agg, values):
    ...

In [ ]:
check("05-iterators/running-agg")

<details>
<summary><b>hints</b></summary>

1. The shape is `while True:` wrapping `try: value = yield state` with `except ValueError: continue` attached to that `try`.
2. Build the state dict from a small closure so you never forget a key.
3. `bool` is a subclass of `int`, so `isinstance(True, (int, float))` is True — you need an explicit `not isinstance(value, bool)`.
4. Wrap the whole `while` in `try: ... finally: CLOSED_WITH.append(count)`; `close()` raises GeneratorExit at the yield and the finally runs.
5. `feed` can check `inspect.getgeneratorstate(agg) == 'GEN_CREATED'` to decide whether to prime.

</details>

<details>
<summary><b>reference solution</b></summary>

```python
CLOSED_WITH = []

def aggregator():
    count = total = 0
    lo = hi = None
    error = None

    def state():
        return {
            "count": count,
            "total": total,
            "mean": total / count if count else None,
            "min": lo,
            "max": hi,
            "error": error,
        }

    try:
        while True:
            try:
                value = yield state()
            except ValueError:
                continue              # absorbed: loop re-yields state
            if value == "reset":
                count = total = 0
                lo = hi = None
                error = None
                continue
            if isinstance(value, bool) or not isinstance(value, (int, float)):
                error = f"cannot aggregate {value!r}"
                continue              # statistics deliberately untouched
            error = None
            count += 1
            total += value
            lo = value if lo is None else min(lo, value)
            hi = value if hi is None else max(hi, value)
    finally:
        CLOSED_WITH.append(count)

def feed(agg, values):
    import inspect
    if inspect.getgeneratorstate(agg) == "GEN_CREATED":
        next(agg)
    last = None
    for value in values:
        last = agg.send(value)
    return last
```

</details>

## Takeaways

- `iter()` + `__next__()` + `StopIteration` is the whole protocol.
- Separate the iterable from the iterator to stay re-iterable.
- A generator function returns a generator; the body runs lazily.
- `send`/`throw`/`close` make generators two-way; `yield from`
  forwards all three and captures the return value.
- Pipelines of generators stream in constant memory.

---

### ✅ Lesson complete

Exercises in this lesson: **2**. Check your standing across the whole course with:

```
uv run learn.py list
```